# Fine-tune LLM with Ray TorchTrainer on Amazon SageMaker HyperPod

**FSDP + LoRA/QLoRA on a persistent Ray cluster (EKS + KubeRay)**

This notebook demonstrates how to run a Ray-based distributed training job on **SageMaker HyperPod** with EKS orchestration. Unlike ephemeral SageMaker Training Jobs, HyperPod provides a persistent GPU cluster where Ray runs continuously — you submit jobs to it rather than spinning up new instances each time.

### Key differences from SageMaker Training Jobs approach

| Aspect | SageMaker Training Jobs | SageMaker HyperPod |
|--------|------------------------|--------------------|
| **Compute** | Ephemeral (spins up per job) | Persistent cluster |
| **Ray cluster** | Bootstrapped via launcher.py | Always running (KubeRay) |
| **Job submission** | `ModelTrainer.train()` | `ray job submit` |
| **Data access** | S3 → `/opt/ml/input/data/` | FSx for Lustre (`/mnt/fsx`) |
| **Orchestration** | SageMaker managed | EKS + KubeRay operator |
| **Multi-tenancy** | One job per cluster | Multiple jobs, namespaces, priorities |
| **Node health** | Job-scoped | HyperPod auto-resume + health checks |
| **Cost model** | Pay per job duration | Reserved capacity (ODCR), higher utilization |

### Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                  SageMaker HyperPod Cluster                 │
│  ┌───────────────────────────────────────────────────────┐  │
│  │               Amazon EKS Control Plane                │  │
│  │  ┌─────────────┐  ┌──────────────┐  ┌─────────────┐  │  │
│  │  │   KubeRay    │  │  Ray Head    │  │ Prometheus  │  │  │
│  │  │  Operator    │  │  (no GPU)    │  │  + Grafana  │  │  │
│  │  └─────────────┘  └──────┬───────┘  └─────────────┘  │  │
│  │                          │                            │  │
│  │         ┌────────────────┼────────────────┐           │  │
│  │         ▼                ▼                ▼           │  │
│  │  ┌────────────┐  ┌────────────┐  ┌────────────┐      │  │
│  │  │ Ray Worker │  │ Ray Worker │  │ Ray Worker │ ...  │  │
│  │  │ 4x A10G    │  │ 4x A10G    │  │ 4x A10G    │      │  │
│  │  │ g5.12xl    │  │ g5.12xl    │  │ g5.12xl    │      │  │
│  │  └────────────┘  └────────────┘  └────────────┘      │  │
│  └───────────────────────────────────────────────────────┘  │
│                          │                                  │
│                ┌─────────▼─────────┐                        │
│                │  FSx for Lustre   │                        │
│                │  /mnt/fsx         │                        │
│                │  datasets, ckpts  │                        │
│                └───────────────────┘                        │
└─────────────────────────────────────────────────────────────┘
```

## Prerequisites

Before running this notebook, ensure:

1. **HyperPod cluster** is created with EKS orchestration and the **Ray addon** enabled
2. **FSx for Lustre** file system is created and the **FSx CSI driver** (`fsx.csi.aws.com`) is installed on EKS
3. **SageMaker execution role** has EKS permissions (`eks:DescribeCluster`, `eks:ListClusters`) and an EKS access entry with cluster admin policy

### What HyperPod with Ray addon auto-creates

When you create a HyperPod cluster with the Ray addon, it automatically provisions:
- `ray-training` namespace
- `ray-training-sa` service account
- KubeRay operator (in `ray-system` namespace)
- FSx PersistentVolume (`fsx-pv`) bound to `default/fsx-claim`

This notebook handles the remaining setup: creating a dedicated FSx PVC for the `ray-training` namespace, verifying the KubeRay operator, and deploying the Ray cluster.

## Step 1: Install Dependencies

In [ ]:
!pip install -q ray[client]==2.56.1 datasets transformers huggingface_hub pyyaml boto3

In [ ]:
# Install kubectl (skip if already installed)
import subprocess, sys, platform

try:
    result = subprocess.run(["kubectl", "version", "--client", "--short"], capture_output=True, text=True)
    print(f"kubectl already installed: {result.stdout.strip()}")
except FileNotFoundError:
    print("Installing kubectl...")
    system = platform.system().lower()  # linux or darwin
    arch = "amd64" if platform.machine() == "x86_64" else "arm64"
    !curl -sLO "https://dl.k8s.io/release/$(curl -sL https://dl.k8s.io/release/stable.txt)/bin/{system}/{arch}/kubectl"
    !chmod +x kubectl
    !sudo mv kubectl /usr/local/bin/kubectl
    !kubectl version --client --short
    print("kubectl installed successfully.")

In [ ]:
# Install Helm (skip if already installed)
import subprocess, platform

try:
    result = subprocess.run(["helm", "version", "--short"], capture_output=True, text=True)
    print(f"Helm already installed: {result.stdout.strip()}")
except FileNotFoundError:
    print("Installing Helm...")
    !curl -fsSL https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash
    !helm version --short
    print("Helm installed successfully.")

In [ ]:
# Configure kubectl to connect to the HyperPod EKS cluster
REGION = "us-east-1"
EKS_CLUSTER_NAME = "sagemaker-rayhpcluster-fe57bddd-eks"

# Update kubeconfig to point to the HyperPod EKS cluster
!aws eks update-kubeconfig --name {EKS_CLUSTER_NAME} --region {REGION}

# Verify connectivity
!kubectl get nodes

## Step 2: Set Up the HyperPod Cluster

HyperPod with the Ray addon auto-creates the `ray-training` namespace, service account, and KubeRay operator. However, the FSx PersistentVolume (`fsx-pv`) is bound to `default/fsx-claim` and cannot be shared with the `ray-training` namespace. We need to create a dedicated PV/PVC for `ray-training`.

In [ ]:
# Discover FSx details from the existing PV created by HyperPod
import subprocess, json

result = subprocess.run(
    ["kubectl", "get", "pv", "fsx-pv", "-o", "json"],
    capture_output=True, text=True
)
pv = json.loads(result.stdout)
csi = pv["spec"]["csi"]

FSX_VOLUME_HANDLE = csi["volumeHandle"]
FSX_DNS = csi["volumeAttributes"]["dnsname"]
FSX_MOUNT_NAME = csi["volumeAttributes"]["mountname"]

print(f"FSx Volume Handle: {FSX_VOLUME_HANDLE}")
print(f"FSx DNS Name:      {FSX_DNS}")
print(f"FSx Mount Name:    {FSX_MOUNT_NAME}")

In [ ]:
# Create a dedicated FSx PV and PVC for the ray-training namespace
# The existing fsx-pv is already bound to default/fsx-claim and cannot be reused

fsx_pv_manifest = f"""
apiVersion: v1
kind: PersistentVolume
metadata:
  name: fsx-pv-ray
spec:
  capacity:
    storage: 1200Gi
  volumeMode: Filesystem
  accessModes:
    - ReadWriteMany
  mountOptions:
    - flock
  persistentVolumeReclaimPolicy: Retain
  storageClassName: fsx-sc
  claimRef:
    namespace: ray-training
    name: fsx-pvc
  csi:
    driver: fsx.csi.aws.com
    volumeHandle: {FSX_VOLUME_HANDLE}
    volumeAttributes:
      dnsname: {FSX_DNS}
      mountname: {FSX_MOUNT_NAME}
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: fsx-pvc
  namespace: ray-training
spec:
  accessModes:
    - ReadWriteMany
  storageClassName: fsx-sc
  resources:
    requests:
      storage: 1200Gi
  volumeName: fsx-pv-ray
"""

with open("/tmp/fsx-ray-pv.yaml", "w") as f:
    f.write(fsx_pv_manifest)

!kubectl apply -f /tmp/fsx-ray-pv.yaml
!kubectl get pv,pvc -n ray-training

In [ ]:
# Verify KubeRay operator is running (auto-installed by HyperPod Ray addon)
# If it's in CrashLoopBackOff, reinstall with: helm uninstall kuberay-operator -n ray-system
# then: helm install kuberay-operator kuberay/kuberay-operator --namespace ray-system
!kubectl get pods -n ray-system

In [ ]:
# Deploy the Ray cluster
!kubectl apply -f ray-cluster.yaml

In [ ]:
# Wait for the Ray cluster to be ready
!kubectl -n ray-training wait --for=condition=Ready pod -l ray-node-type=head --timeout=300s
print("Ray head node is ready.")

!kubectl -n ray-training get pods -l app=ray-training

## Step 3: Prepare the Dataset

We use the same dataset as the original notebook — `FreedomIntelligence/medical-o1-reasoning-SFT` — but write it to FSx for Lustre instead of S3.

In [ ]:
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen3-0.6B"
DATASET_ID = "FreedomIntelligence/medical-o1-reasoning-SFT"
NUM_SAMPLES = 1000

dataset = load_dataset(DATASET_ID, "en", split=f"train[:{NUM_SAMPLES}]")
print(f"Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nExample:\n{dataset[0]}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

SYSTEM_PROMPT = """You are a medical reasoning assistant. Respond to user queries
with clear, step-by-step medical reasoning."""

def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["Question"]},
        {"role": "assistant", "content": f"<think>\n{example['Complex_CoT']}\n</think>\n\n{example['Response']}"},
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example

dataset = dataset.map(format_chat)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Write datasets to FSx via a pod (exec into the head node)
# First, save locally as JSON
import json, os

os.makedirs("data", exist_ok=True)
train_dataset.to_json("data/train.json")
val_dataset.to_json("data/val.json")

print("Datasets saved locally.")

In [ ]:
# Get the Ray head pod name
import subprocess

result = subprocess.run(
    ["kubectl", "-n", "ray-training", "get", "pods", "-l", "ray-node-type=head",
     "-o", "jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True
)
HEAD_POD = result.stdout.strip()
print(f"Ray head pod: {HEAD_POD}")

In [ ]:
# Create dataset directories on FSx and copy data
!kubectl -n ray-training exec {HEAD_POD} -- mkdir -p /mnt/fsx/datasets/medical-o1/train
!kubectl -n ray-training exec {HEAD_POD} -- mkdir -p /mnt/fsx/datasets/medical-o1/val
!kubectl -n ray-training exec {HEAD_POD} -- mkdir -p /mnt/fsx/checkpoints/qwen3-medical
!kubectl -n ray-training exec {HEAD_POD} -- mkdir -p /mnt/fsx/output/qwen3-medical

!kubectl cp data/train.json ray-training/{HEAD_POD}:/mnt/fsx/datasets/medical-o1/train/train.json
!kubectl cp data/val.json ray-training/{HEAD_POD}:/mnt/fsx/datasets/medical-o1/val/val.json

print("Datasets copied to FSx.")

## Step 4: Port-Forward to the Ray Dashboard

Open a terminal and run:
```bash
kubectl -n ray-training port-forward svc/ray-hyperpod-training-head-svc 8265:8265
```

Then open http://localhost:8265 to view the Ray Dashboard.

In [ ]:
# Verify Ray cluster is accessible
RAY_ADDRESS = "http://localhost:8265"

from ray.job_submission import JobSubmissionClient

client = JobSubmissionClient(RAY_ADDRESS)
jobs = client.list_jobs()
print(f"Ray Dashboard: {RAY_ADDRESS}")
print(f"Connected successfully. Existing jobs: {len(jobs)}")

## Step 5: Upload Training Script and Config

Copy the training script and config to FSx so Ray workers can access them.

In [ ]:
# Copy training script and config to FSx
!kubectl -n ray-training exec {HEAD_POD} -- mkdir -p /mnt/fsx/scripts

!kubectl cp train_ray.py ray-training/{HEAD_POD}:/mnt/fsx/scripts/train_ray.py
!kubectl cp args.yaml ray-training/{HEAD_POD}:/mnt/fsx/scripts/args.yaml
!kubectl cp requirements.txt ray-training/{HEAD_POD}:/mnt/fsx/scripts/requirements.txt

print("Training files copied to FSx.")

In [ ]:
# Install dependencies on all Ray nodes
!kubectl -n ray-training exec {HEAD_POD} -- pip install -q -r /mnt/fsx/scripts/requirements.txt

## Step 6: Submit the Training Job

Two options:
1. **Ray Jobs API** (recommended) — submit via the dashboard/client
2. **RayJob CRD** — submit via Kubernetes manifest

### Option A: Ray Jobs API

In [ ]:
from ray.job_submission import JobSubmissionClient

client = JobSubmissionClient(RAY_ADDRESS)

job_id = client.submit_job(
    entrypoint="python /mnt/fsx/scripts/train_ray.py --config /mnt/fsx/scripts/args.yaml",
    runtime_env={
        "working_dir": "/mnt/fsx/scripts",
        "env_vars": {
            "HF_HOME": "/mnt/fsx/huggingface",
            "NCCL_DEBUG": "INFO",
            "NCCL_SOCKET_IFNAME": "eth0",
        },
    },
    metadata={
        "name": "qwen3-medical-finetune",
        "cluster": "hyperpod",
    },
)

print(f"Job submitted: {job_id}")
print(f"Dashboard: {RAY_ADDRESS}/#/jobs/{job_id}")

In [ ]:
# Monitor job status
import time

while True:
    status = client.get_job_status(job_id)
    print(f"Status: {status}")
    if status.is_terminal():
        break
    time.sleep(30)

# Print logs
logs = client.get_job_logs(job_id)
print(f"\n{'='*60}\nJob Logs (last 2000 chars):\n{'='*60}")
print(logs[-2000:])

### Option B: RayJob Kubernetes CRD

Alternatively, submit the job as a Kubernetes resource. This is useful for CI/CD pipelines and GitOps workflows.

In [ ]:
rayjob_manifest = """
apiVersion: ray.io/v1
kind: RayJob
metadata:
  name: qwen3-medical-finetune
  namespace: ray-training
spec:
  entrypoint: "python /mnt/fsx/scripts/train_ray.py --config /mnt/fsx/scripts/args.yaml"
  runtimeEnvYAML: |
    working_dir: /mnt/fsx/scripts
    env_vars:
      HF_HOME: /mnt/fsx/huggingface
      NCCL_DEBUG: INFO
  shutdownAfterJobFinishes: false
  clusterSelector:
    ray.io/cluster: ray-hyperpod-training
  submitterPodTemplate:
    spec:
      containers:
        - name: submitter
          image: rayproject/ray:2.56.1-py311
          resources:
            limits:
              cpu: "1"
              memory: "2Gi"
"""

with open("rayjob.yaml", "w") as f:
    f.write(rayjob_manifest)

print("RayJob manifest written. Apply with:")
print("  kubectl apply -f rayjob.yaml")

In [ ]:
# Apply the RayJob (uncomment to use this approach instead of Option A)
# !kubectl apply -f rayjob.yaml
# !kubectl -n ray-training get rayjob qwen3-medical-finetune -w

## Step 7: Verify Output

After training completes, the model artifacts are on FSx.

In [ ]:
# List output artifacts on FSx
!kubectl -n ray-training exec {HEAD_POD} -- ls -la /mnt/fsx/output/qwen3-medical/
!kubectl -n ray-training exec {HEAD_POD} -- ls -la /mnt/fsx/output/qwen3-medical/final_model/ 2>/dev/null || echo "Final model not yet saved"
!kubectl -n ray-training exec {HEAD_POD} -- ls -la /mnt/fsx/output/qwen3-medical/merged_model/ 2>/dev/null || echo "Merged model not yet saved"

In [ ]:
# Copy the merged model to S3 for deployment
S3_BUCKET = "your-model-bucket"  # <-- Replace
MODEL_S3_PATH = f"s3://{S3_BUCKET}/models/qwen3-medical-finetuned/"

!kubectl -n ray-training exec {HEAD_POD} -- \
    aws s3 sync /mnt/fsx/output/qwen3-medical/merged_model/ {MODEL_S3_PATH}

print(f"Model uploaded to: {MODEL_S3_PATH}")

## Step 8: Clean Up

Delete the Ray cluster when done (HyperPod nodes remain available for other workloads).

In [ ]:
# Delete the Ray cluster (nodes return to HyperPod pool)
# !kubectl delete -f ray-cluster.yaml

# Or scale down workers to 0 (keep head for dashboard)
# !kubectl -n ray-training patch raycluster ray-hyperpod-training \
#     --type merge -p '{"spec":{"workerGroupSpecs":[{"groupName":"gpu-workers","replicas":0}]}}'

print("Cleanup commands above are commented out for safety. Uncomment to execute.")

## Summary

| Step | What happened |
|------|---------------|
| 1 | Installed kubectl, Helm, and configured kubeconfig for HyperPod EKS |
| 2 | Created FSx PV/PVC for `ray-training` namespace and deployed RayCluster |
| 3 | Prepared and copied dataset to FSx for Lustre |
| 4 | Submitted Ray training job (FSDP + LoRA) |
| 5 | Model saved to FSx, synced to S3 for deployment |

### HyperPod advantages over Training Jobs for Ray workloads

- **No cold start** — Ray cluster is always running, job starts immediately
- **Shared FSx** — datasets, checkpoints, and models persist across jobs (no S3 upload/download per job)
- **Auto-resume** — HyperPod replaces failed nodes and Ray recovers automatically
- **Task fungibility** — same nodes can run training, inference, or notebooks
- **Multi-tenancy** — multiple teams can share the cluster with K8s namespaces and priorities
- **Interactive debugging** — Ray Dashboard always accessible, `ray attach` for live debugging